In [7]:
# !pip install scikit-optimize

### Logistic Regression

In [8]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

### 1. 사전 작업


In [9]:
# ADASYN 적용된 학습 데이터 로드
train_df = pd.read_csv(r"../../data/processed/ADASYN/data_train_adasyn.csv")

print("shape:", train_df.shape)
print(train_df.head())
print("\ncolumns:")
print(train_df.columns.tolist())

# 실제 타깃 컬럼명
target_col = "Risk_Label"

# 전체 데이터(X)와 타깃(y) 분리
# 주의: X에 Risk_Label이 포함되면 데이터 누수가 생기므로 반드시 제거
X = train_df.drop(columns=[target_col]).copy()
y = train_df[target_col].astype(int)

print("\nX shape:", X.shape)
print("y distribution:")
print(y.value_counts())
print(y.value_counts(normalize=True))

# 불균형 데이터 평가를 위한 G-Mean 및 H1 함수
# H1 = F1-score와 G-Mean의 조화평균
def gmean_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return np.sqrt(recall * specificity)


def h1_score(y_true, y_pred):
    f1 = f1_score(y_true, y_pred, zero_division=0)
    gmean = gmean_score(y_true, y_pred)

    return 2 * f1 * gmean / (f1 + gmean) if (f1 + gmean) > 0 else 0.0


shape: (2466, 10)
   NASDAQ_return(%)  Brent Crude Oil_return(%)  Gold Spot_return(%)  \
0          0.593612                   0.545323             0.488488   
1          0.259590                   0.545323             0.696235   
2          0.758918                   0.545323             0.535190   
3          0.592042                   0.545323             0.630347   
4          0.610832                   0.545323             0.657784   

   KOSDAQ_return(%)  KOSPI 200 Volume  KOSPI 200 lagged_1_return(%)  \
0          0.349814          0.000817                      0.593898   
1          0.617084          0.000552                      0.548072   
2          0.580751          0.000633                      0.616220   
3          0.668308          0.000972                      0.551119   
4          0.566195          0.000917                      0.688352   

   KOSPI 200_OG  KOSPI 200_PPO  VKOSPI_Change(%)  Risk_Label  
0      0.728036       1.000000          0.298311           0  
1 

### 2. Valid 기반 파라미터 최적화

In [10]:
# =========================
# 2. Valid 기반 LR 하이퍼파라미터 + cutoff 동시 탐색
#    선택 기준: H1 = F1-score와 G-Mean의 조화평균
# =========================
import os
import numpy as np
import pandas as pd
import sklearn
from packaging import version

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    log_loss, roc_auc_score, average_precision_score
)

# -------------------------
# 1) 데이터 로드
# -------------------------
train_df = pd.read_csv(r"../../data/processed/ADASYN/data_train_adasyn.csv")
valid_df = pd.read_csv(r"../../data/processed/ADASYN/data_valid.csv")

target_col = "Risk_Label"

# X에 target이 들어가면 데이터 누수이므로 반드시 제거
X_train_raw = train_df.drop(columns=[target_col]).copy()
y_train = train_df[target_col].astype(int)

X_valid_raw = valid_df.drop(columns=[target_col]).copy()
y_valid = valid_df[target_col].astype(int)

# valid/test에서 컬럼 불일치가 있으면 0으로 채우지 말고 오류 내는 게 안전함
missing_in_valid = set(X_train_raw.columns) - set(X_valid_raw.columns)
extra_in_valid = set(X_valid_raw.columns) - set(X_train_raw.columns)

if missing_in_valid or extra_in_valid:
    raise ValueError(
        f"train-valid 컬럼 불일치\n"
        f"valid에 없는 train 컬럼: {missing_in_valid}\n"
        f"valid에만 있는 컬럼: {extra_in_valid}"
    )

X_valid_raw = X_valid_raw[X_train_raw.columns]

X_train_model = X_train_raw.values
X_valid_model = X_valid_raw.values
feature_names = X_train_raw.columns

print("Train shape:", X_train_model.shape)
print("Valid shape:", X_valid_model.shape)
print("\nTrain class distribution")
print(y_train.value_counts(normalize=True))
print("\nValid class distribution")
print(y_valid.value_counts(normalize=True))


# -------------------------
# 2) 평가 함수
# -------------------------
def get_binary_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    gmean = np.sqrt(rec * specificity)

    # H1 = F1-score와 G-Mean의 조화평균
    h1 = 2 * f1 * gmean / (f1 + gmean) if (f1 + gmean) > 0 else 0.0

    try:
        ll = log_loss(y_true, y_prob, labels=[0, 1])
    except Exception:
        ll = np.nan

    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except Exception:
        roc_auc = np.nan

    try:
        pr_auc = average_precision_score(y_true, y_prob)
    except Exception:
        pr_auc = np.nan

    return {
        "threshold": threshold,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "specificity": specificity,
        "f1": f1,
        "gmean": gmean,
        "h1": h1,
        "log_loss": ll,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }


def make_threshold_grid(valid_prob):
    # 0.01 간격 cutoff + validation 확률 분위수 기반 cutoff를 함께 사용
    # 확률이 특정 구간에 몰려 있어도 후보 cutoff가 충분히 잡히도록 함
    return np.unique(np.r_[
        np.arange(0.01, 0.991, 0.01),
        np.quantile(valid_prob, np.linspace(0.01, 0.99, 99))
    ])


def find_best_cutoff(y_valid, valid_prob):
    threshold_history = []

    for thr in make_threshold_grid(valid_prob):
        m = get_binary_metrics(y_valid, valid_prob, threshold=thr)
        threshold_history.append({
            "threshold": float(thr),
            "valid_accuracy": m["accuracy"],
            "valid_precision": m["precision"],
            "valid_recall": m["recall"],
            "valid_specificity": m["specificity"],
            "valid_f1": m["f1"],
            "valid_gmean": m["gmean"],
            "valid_h1": m["h1"],
            "valid_log_loss": m["log_loss"],
            "valid_roc_auc": m["roc_auc"],
            "valid_pr_auc": m["pr_auc"],
            "tn": m["tn"],
            "fp": m["fp"],
            "fn": m["fn"],
            "tp": m["tp"]
        })

    threshold_df = pd.DataFrame(threshold_history)

    # cutoff 선택 기준:
    # 1순위 H1 최대
    # 2순위 G-Mean 최대
    # 3순위 F1 최대
    # 4순위 Recall 최대
    threshold_df = threshold_df.sort_values(
        by=["valid_h1", "valid_gmean", "valid_f1", "valid_recall"],
        ascending=[False, False, False, False]
    ).reset_index(drop=True)

    return threshold_df.iloc[0], threshold_df


# -------------------------
# 3) sklearn 버전 호환 LR 생성 함수
# -------------------------
def make_lr(C, l1_ratio, max_iter=5000, random_state=1):
    kwargs = dict(
        solver="saga",
        C=C,
        l1_ratio=l1_ratio,
        max_iter=max_iter,
        random_state=random_state,
        tol=1e-4
    )

    # sklearn 1.8부터 penalty 경고가 뜨므로 버전별 처리
    if version.parse(sklearn.__version__) < version.parse("1.8"):
        kwargs["penalty"] = "elasticnet"

    return LogisticRegression(**kwargs)


# -------------------------
# 4) 1차 coarse grid search
#    C, l1_ratio, cutoff를 valid set에서 동시에 선택
# -------------------------
C_grid_1 = np.logspace(-4, 4, 33)
l1_grid_1 = np.round(np.linspace(0.0, 1.0, 21), 2)

search_history_1 = []
threshold_history_all_1 = []

print(f"\n[1차 탐색] LR 조합 수: {len(C_grid_1) * len(l1_grid_1)}")
print("각 LR 조합마다 valid set에서 cutoff를 함께 탐색합니다.")

for C_val in C_grid_1:
    for l1_val in l1_grid_1:
        model_trial = make_lr(C=C_val, l1_ratio=l1_val)
        model_trial.fit(X_train_model, y_train)

        valid_prob = model_trial.predict_proba(X_valid_model)[:, 1]
        best_thr_row, threshold_df_tmp = find_best_cutoff(y_valid, valid_prob)

        search_history_1.append({
            "stage": "coarse",
            "C": C_val,
            "l1_ratio": l1_val,
            "best_cutoff": best_thr_row["threshold"],
            "valid_accuracy": best_thr_row["valid_accuracy"],
            "valid_precision": best_thr_row["valid_precision"],
            "valid_recall": best_thr_row["valid_recall"],
            "valid_specificity": best_thr_row["valid_specificity"],
            "valid_f1": best_thr_row["valid_f1"],
            "valid_gmean": best_thr_row["valid_gmean"],
            "valid_h1": best_thr_row["valid_h1"],
            "valid_log_loss": best_thr_row["valid_log_loss"],
            "valid_roc_auc": best_thr_row["valid_roc_auc"],
            "valid_pr_auc": best_thr_row["valid_pr_auc"],
            "tn": best_thr_row["tn"],
            "fp": best_thr_row["fp"],
            "fn": best_thr_row["fn"],
            "tp": best_thr_row["tp"]
        })

        threshold_df_tmp = threshold_df_tmp.copy()
        threshold_df_tmp["stage"] = "coarse"
        threshold_df_tmp["C"] = C_val
        threshold_df_tmp["l1_ratio"] = l1_val
        threshold_history_all_1.append(threshold_df_tmp)

search_df_1 = pd.DataFrame(search_history_1)

search_df_1 = search_df_1.sort_values(
    by=["valid_h1", "valid_gmean", "valid_f1", "valid_recall", "valid_log_loss"],
    ascending=[False, False, False, False, True]
).reset_index(drop=True)

print("\n[1차 탐색 상위 10개 - H1 기준]")
print(search_df_1.head(10))

best_1 = search_df_1.iloc[0]
best_C_1 = float(best_1["C"])
best_l1_1 = float(best_1["l1_ratio"])

print("\n[1차 Best - H1 기준]")
print(f"C={best_C_1:.8f}, l1_ratio={best_l1_1:.4f}, cutoff={best_1['best_cutoff']:.4f}, "
      f"H1={best_1['valid_h1']:.4f}, G-Mean={best_1['valid_gmean']:.4f}, "
      f"F1={best_1['valid_f1']:.4f}, Recall={best_1['valid_recall']:.4f}")


Train shape: (2466, 9)
Valid shape: (1438, 9)

Train class distribution
Risk_Label
0    0.675182
1    0.324818
Name: proportion, dtype: float64

Valid class distribution
Risk_Label
0    0.887344
1    0.112656
Name: proportion, dtype: float64

[1차 탐색] LR 조합 수: 693
각 LR 조합마다 valid set에서 cutoff를 함께 탐색합니다.

[1차 탐색 상위 10개 - H1 기준]
    stage         C  l1_ratio  best_cutoff  valid_accuracy  valid_precision  \
0  coarse  1.000000      0.00     0.389166        0.799722         0.281250   
1  coarse  1.000000      0.25     0.390000        0.797636         0.278351   
2  coarse  0.562341      0.40     0.390000        0.800417         0.280702   
3  coarse  1.778279      0.10     0.390000        0.794159         0.275168   
4  coarse  1.000000      0.55     0.390000        0.794159         0.275168   
5  coarse  1.000000      0.30     0.390000        0.796940         0.277397   
6  coarse  1.778279      0.05     0.390000        0.793463         0.274247   
7  coarse  1.000000      0.35     0.3900

In [11]:
# -------------------------
# 5) 2차 local refinement
#    1차 best 주변에서 C, l1_ratio, cutoff 동시 재탐색
#    선택 기준: H1 = F1-score와 G-Mean의 조화평균
# -------------------------
C_grid_2 = best_C_1 * np.logspace(-0.5, 0.5, 15)
l1_min = max(0.0, best_l1_1 - 0.20)
l1_max = min(1.0, best_l1_1 + 0.20)
l1_grid_2 = np.round(np.linspace(l1_min, l1_max, 15), 3)

search_history_2 = []
threshold_history_all_2 = []

print(f"\n[2차 세부 탐색] LR 조합 수: {len(C_grid_2) * len(l1_grid_2)}")
print("각 LR 조합마다 valid set에서 cutoff를 함께 탐색합니다.")

for C_val in C_grid_2:
    for l1_val in l1_grid_2:
        model_trial = make_lr(C=C_val, max_iter=5000, l1_ratio=l1_val)
        model_trial.fit(X_train_model, y_train)

        valid_prob = model_trial.predict_proba(X_valid_model)[:, 1]
        best_thr_row, threshold_df_tmp = find_best_cutoff(y_valid, valid_prob)

        search_history_2.append({
            "stage": "refine",
            "C": C_val,
            "l1_ratio": l1_val,
            "best_cutoff": best_thr_row["threshold"],
            "valid_accuracy": best_thr_row["valid_accuracy"],
            "valid_precision": best_thr_row["valid_precision"],
            "valid_recall": best_thr_row["valid_recall"],
            "valid_specificity": best_thr_row["valid_specificity"],
            "valid_f1": best_thr_row["valid_f1"],
            "valid_gmean": best_thr_row["valid_gmean"],
            "valid_h1": best_thr_row["valid_h1"],
            "valid_log_loss": best_thr_row["valid_log_loss"],
            "valid_roc_auc": best_thr_row["valid_roc_auc"],
            "valid_pr_auc": best_thr_row["valid_pr_auc"],
            "tn": best_thr_row["tn"],
            "fp": best_thr_row["fp"],
            "fn": best_thr_row["fn"],
            "tp": best_thr_row["tp"]
        })

        threshold_df_tmp = threshold_df_tmp.copy()
        threshold_df_tmp["stage"] = "refine"
        threshold_df_tmp["C"] = C_val
        threshold_df_tmp["l1_ratio"] = l1_val
        threshold_history_all_2.append(threshold_df_tmp)

search_df_2 = pd.DataFrame(search_history_2)

search_df = pd.concat([search_df_1, search_df_2], ignore_index=True)

search_df = search_df.sort_values(
    by=["valid_h1", "valid_gmean", "valid_f1", "valid_recall", "valid_log_loss"],
    ascending=[False, False, False, False, True]
).reset_index(drop=True)

best_row = search_df.iloc[0]
best_lr_params = {
    "C": float(best_row["C"]),
    "l1_ratio": float(best_row["l1_ratio"])
}
best_cutoff = float(best_row["best_cutoff"])

print("\n[최종 Best Hyperparameters + Cutoff - H1 기준]")
print(f"C        : {best_lr_params['C']:.8f}")
print(f"l1_ratio : {best_lr_params['l1_ratio']:.4f}")
print(f"cutoff   : {best_cutoff:.4f}")
print(f"H1       : {best_row['valid_h1']:.4f}")
print(f"G-Mean   : {best_row['valid_gmean']:.4f}")
print(f"F1       : {best_row['valid_f1']:.4f}")
print(f"Recall   : {best_row['valid_recall']:.4f}")
print(f"Specificity: {best_row['valid_specificity']:.4f}")
print(f"Precision: {best_row['valid_precision']:.4f}")
print(f"LogLoss  : {best_row['valid_log_loss']:.4f}")

print("\n[전체 탐색 상위 15개 - H1 기준]")
print(search_df.head(15))



# -------------------------
# 6) 최종 모델 학습
# -------------------------
model = make_lr(
    C=best_lr_params["C"],
    l1_ratio=best_lr_params["l1_ratio"]
)
model.fit(X_train_model, y_train)



# -------------------------
# 7) 최종 LR 계수 확인
# -------------------------

coef = model.coef_.ravel()

# 계수와 feature 이름 개수 일치 확인
# X_train_model이 DataFrame이면 그 컬럼명을 쓰고,
# ndarray이면 X_train_raw 컬럼명을 사용
if hasattr(X_train_model, "columns"):
    feature_names = X_train_model.columns
else:
    feature_names = X_train_raw.columns

if len(coef) != len(feature_names):
    raise ValueError(
        f"계수 개수({len(coef)})와 feature 개수({len(feature_names)})가 다릅니다. "
        "X_train_model과 feature_names가 같은 변수 순서를 갖는지 확인하세요."
    )

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": coef,
    "abs_coef": np.abs(coef)
})

coef_df = coef_df.sort_values("abs_coef", ascending=False).reset_index(drop=True)

# saga solver는 수치 오차가 있을 수 있으므로 아주 작은 값은 0으로 간주
zero_tol = 1e-8
coef_df["nonzero_coef"] = coef_df["abs_coef"] > zero_tol

nonzero_coef_df = coef_df[coef_df["nonzero_coef"]].copy().reset_index(drop=True)
zero_coef_df = coef_df[~coef_df["nonzero_coef"]].copy().reset_index(drop=True)

print("\n" + "=" * 80)
print("[최종 LR Hyperparameters - H1 기준]")
print(f"Final C        : {best_lr_params['C']:.8f}")
print(f"Final l1_ratio : {best_lr_params['l1_ratio']:.4f}")
print(f"Final cutoff   : {best_cutoff:.4f}")
print(f"Final valid H1 : {best_row['valid_h1']:.4f}")

print("\n[최종 LR 계수 확인]")
print(f"전체 변수 개수        : {len(coef_df)}")
print(f"계수가 0이 아닌 변수 수: {len(nonzero_coef_df)}")
print(f"계수가 0인 변수 수     : {len(zero_coef_df)}")

if best_lr_params["l1_ratio"] <= zero_tol:
    print("\n주의: 최종 l1_ratio가 0에 가까우므로 이 모델은 사실상 L2/Ridge 성격의 LR입니다.")
    print("따라서 아래 계수는 'LR 변수선택 결과'가 아니라, 변수별 계수 크기 확인용으로 해석하세요.")
else:
    print("\n주의: l1_ratio > 0이므로 일부 계수가 0으로 수축될 수 있습니다.")
    print("다만 이는 사전 변수선택 이후 LR 내부 정규화에 따른 추가 수축 결과입니다.")

print("\n[계수 절댓값 기준 상위 변수]")
display(nonzero_coef_df)

if len(zero_coef_df) > 0:
    print("\n[계수가 0으로 나온 변수]")
    display(zero_coef_df)
else:
    print("\n[계수가 0으로 나온 변수 없음]")

print("\n" + "=" * 30 + " 최종 LR 모델 학습 완료 " + "=" * 30)



[2차 세부 탐색] LR 조합 수: 225
각 LR 조합마다 valid set에서 cutoff를 함께 탐색합니다.

[최종 Best Hyperparameters + Cutoff - H1 기준]
C        : 1.00000000
l1_ratio : 0.0000
cutoff   : 0.3892
H1       : 0.4627
G-Mean   : 0.6472
F1       : 0.3600
Recall   : 0.5000
Specificity: 0.8378
Precision: 0.2812
LogLoss  : 0.4374

[전체 탐색 상위 15개 - H1 기준]
     stage         C  l1_ratio  best_cutoff  valid_accuracy  valid_precision  \
0   coarse  1.000000     0.000     0.389166        0.799722         0.281250   
1   refine  1.000000     0.000     0.389166        0.799722         0.281250   
2   refine  1.178769     0.000     0.390000        0.798331         0.279310   
3   refine  1.178769     0.014     0.390000        0.798331         0.279310   
4   refine  1.178769     0.029     0.390000        0.798331         0.279310   
5   refine  1.178769     0.043     0.390000        0.798331         0.279310   
6   refine  1.178769     0.057     0.390000        0.797636         0.278351   
7   refine  1.178769     0.071     0.3900

,feature,coef,abs_coef,nonzero_coef
0,NASDAQ_return(%),-6.709401,6.709401,True
1,KOSPI 200_OG,-2.894938,2.894938,True
2,VKOSPI_Change(%),-1.600908,1.600908,True
3,KOSPI 200 lagged_1_return(%),-1.592016,1.592016,True
4,KOSPI 200 Volume,1.478120,1.478120,True
5,KOSPI 200_PPO,-0.826723,0.826723,True
6,Gold Spot_return(%),-0.758382,0.758382,True
7,KOSDAQ_return(%),0.646928,0.646928,True
8,Brent Crude Oil_return(%),-0.363788,0.363788,True



[계수가 0으로 나온 변수 없음]

============================== 최종 LR 모델 학습 완료 ==============================


### 3. test data 성능 평가


In [12]:
test_df = pd.read_csv(r"../../data/processed/ADASYN/data_test.csv")

X_test_df = test_df.drop(columns=[target_col]).copy()
y_test = test_df[target_col].astype(int)

missing_in_test = set(feature_names) - set(X_test_df.columns)
extra_in_test = set(X_test_df.columns) - set(feature_names)

if missing_in_test or extra_in_test:
    raise ValueError(
        f"train-test 컬럼 불일치\n"
        f"test에 없는 train 컬럼: {missing_in_test}\n"
        f"test에만 있는 컬럼: {extra_in_test}"
    )

X_test_df = X_test_df[feature_names]
X_test = X_test_df.values

y_test_prob = model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= best_cutoff).astype(int)

test_metrics = get_binary_metrics(y_test, y_test_prob, threshold=best_cutoff)

print("\n[TEST performance - H1 기준으로 선택된 cutoff/model]")
print(f"Cutoff    : {best_cutoff:.4f}")
print(f"Accuracy  : {test_metrics['accuracy']:.4f}")
print(f"Precision : {test_metrics['precision']:.4f}")
print(f"Recall    : {test_metrics['recall']:.4f}")
print(f"Specificity: {test_metrics['specificity']:.4f}")
print(f"F1-score  : {test_metrics['f1']:.4f}")
print(f"G-Mean    : {test_metrics['gmean']:.4f}")
print(f"H1        : {test_metrics['h1']:.4f}")
print(f"ROC-AUC   : {test_metrics['roc_auc']:.4f}")
print(f"PR-AUC    : {test_metrics['pr_auc']:.4f}")
print(f"LogLoss   : {test_metrics['log_loss']:.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_test_pred, labels=[0, 1]))

print("\nClassification Report")
print(classification_report(y_test, y_test_pred, digits=4, zero_division=0))



[TEST performance - H1 기준으로 선택된 cutoff/model]
Cutoff    : 0.3892
Accuracy  : 0.8090
Precision : 0.2800
Recall    : 0.4615
Specificity: 0.8523
F1-score  : 0.3485
G-Mean    : 0.6272
H1        : 0.4481
ROC-AUC   : 0.7353
PR-AUC    : 0.2937
LogLoss   : 0.4061

Confusion Matrix
[[623 108]
 [ 49  42]]

Classification Report
              precision    recall  f1-score   support

           0     0.9271    0.8523    0.8881       731
           1     0.2800    0.4615    0.3485        91

    accuracy                         0.8090       822
   macro avg     0.6035    0.6569    0.6183       822
weighted avg     0.8554    0.8090    0.8284       822



In [15]:
# =========================
# 4. LR 예측 결과 저장
#    Date 컬럼 + LR 예측값 컬럼만 저장
# =========================
from pathlib import Path
import pandas as pd
import numpy as np

# 저장 폴더
result_dir = Path(r"../../results/results_ML")
result_dir.mkdir(parents=True, exist_ok=True)

# 현재 test 데이터: Date는 없지만, test 길이 확인용
test_df = pd.read_csv(r"../../data/processed/ADASYN/data_test.csv")

# Date가 남아 있는 원본/최종 데이터 파일
# 네 프로젝트에서 Date 컬럼이 살아있는 파일 경로로 맞추면 됨
date_source_path = Path(r"../../data/processed/data_selected.csv")

date_df = pd.read_csv(date_source_path)

# Date 컬럼 확인
if "Date" not in date_df.columns:
    raise ValueError(
        f"{date_source_path} 파일에도 Date 컬럼이 없습니다. "
        "Date가 남아 있는 원본 데이터 파일 경로로 date_source_path를 바꿔야 합니다."
    )

# 날짜순 정렬
date_df["Date"] = pd.to_datetime(date_df["Date"])
date_df = date_df.sort_values("Date").reset_index(drop=True)

# test set은 chronological split 기준 마지막 구간이므로,
# 예측값 개수만큼 마지막 날짜를 가져옴
n_test = len(y_test_pred)

test_dates = (
    date_df["Date"]
    .tail(n_test)
    .reset_index(drop=True)
    .dt.strftime("%Y-%m-%d")
)

# 길이 확인
if len(test_dates) != len(y_test_pred):
    raise ValueError(
        f"Date 길이({len(test_dates)})와 예측값 길이({len(y_test_pred)})가 다릅니다."
    )

# LR 예측 결과 저장
lr_prediction_result = pd.DataFrame({
    "Date": test_dates,
    "LR_Pred": np.asarray(y_test_pred).ravel()
})

save_path = result_dir / "01. LR_prediction.csv"

lr_prediction_result.to_csv(
    save_path,
    index=False,
    encoding="utf-8-sig"
)

print("LR 예측 결과 저장 완료")
print(save_path)
print(lr_prediction_result.head())

LR 예측 결과 저장 완료
..\..\results\results_ML\01. LR_prediction.csv
         Date  LR_Pred
0  2022-10-17        0
1  2022-10-18        0
2  2022-10-19        1
3  2022-10-20        1
4  2022-10-21        0
